
# CH 5 - 1 : TWO TYPES OF STATISTICAL INFERENCE

## Objectif du notebook

Ce notebook illustre les **deux grands outils de l'inférence statistique** appliqués au trading et au backtesting :

1. **Le test d'hypothèse** : est-ce que mon edge existe vraiment, ou est-ce simplement de la chance ?
2. **L'estimation de paramètre** : si l'edge existe, quelle est sa taille probable et avec quelle incertitude ?

Dans l'esprit EBTA, le trader ne doit pas seulement demander :  
> “Mon backtest est-il profitable ?”

Il doit demander :

> “Mon résultat est-il suffisamment improbable sous l'hypothèse que ma stratégie n'a aucun talent ?”  
> “Et si elle a un avantage, quelle fourchette réaliste puis-je attendre ?”



## 1. L'idée centrale : on ne connaît jamais la vraie performance future

En trading, la **population** représente le monde réel complet : tous les trades futurs possibles, tous les régimes de marché, toutes les conditions non encore observées.

Mais nous n'avons qu'un **échantillon** : le backtest.

Donc on ne connaît pas directement le vrai paramètre :

$$
\mu = \text{rendement moyen futur réel de la stratégie}
$$

On observe seulement une estimation :

$$
\bar{x} = \text{rendement moyen observé dans le backtest}
$$

Le problème est simple mais brutal :

> Un backtest positif peut venir d'un vrai edge... ou d'un coup de chance.

C'est exactement là que l'inférence statistique intervient.



## 2. Les deux questions différentes

### Question 1 — Test d'hypothèse

> Est-ce que l'effet existe ?

Exemple trading :

> La stratégie a-t-elle réellement un rendement moyen supérieur à zéro ?

On formule souvent :

$$
H_0 : \mu \leq 0
$$

La règle n'a aucun talent ou n'a pas de rendement positif réel.

Contre :

$$
H_A : \mu > 0
$$

La règle possède un rendement positif réel.

Le test d'hypothèse donne une décision :

- Rejeter $H_0$
- Ne pas rejeter $H_0$

Il répond donc à une question de type **oui/non**.

---

### Question 2 — Estimation de paramètre

> Quelle est la taille probable de l'effet ?

Exemple trading :

> Si la stratégie a un edge, combien peut-elle raisonnablement rapporter ?

L'estimation donne :

- une **estimation ponctuelle**, par exemple +8 % annualisé ;
- ou mieux, un **intervalle de confiance**, par exemple entre +2 % et +14 % annualisé.

L'estimation répond donc à une question de **quantité**.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

np.random.seed(42)



## 3. Simulation d'un backtest fictif

Imaginons une stratégie de trading avec :

- 252 jours de trading, soit environ une année ;
- un rendement quotidien moyen réel légèrement positif ;
- de la volatilité quotidienne ;
- des rendements journaliers simulés.

Ce n'est pas une preuve de marché réel, c'est un laboratoire pédagogique pour comprendre la logique.


In [ ]:

n_days = 252

# Rendement quotidien réel simulé : 0.04 % par jour environ
true_daily_mean = 0.0004

# Volatilité quotidienne : 1 %
daily_vol = 0.01

returns = np.random.normal(loc=true_daily_mean, scale=daily_vol, size=n_days)

df = pd.DataFrame({
    "daily_return": returns
})

df["equity_curve"] = (1 + df["daily_return"]).cumprod()

sample_mean = df["daily_return"].mean()
sample_std = df["daily_return"].std(ddof=1)
annualized_return = sample_mean * 252
annualized_vol = sample_std * np.sqrt(252)

summary = pd.DataFrame({
    "Mesure": [
        "Moyenne quotidienne observée",
        "Rendement annualisé observé",
        "Volatilité annualisée observée",
        "Nombre de jours"
    ],
    "Valeur": [
        sample_mean,
        annualized_return,
        annualized_vol,
        n_days
    ]
})

summary


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(df["equity_curve"])
plt.title("Courbe d'equity simulée")
plt.xlabel("Jour")
plt.ylabel("Capital relatif")
plt.grid(True)
plt.show()



## 4. Test d'hypothèse : est-ce que le rendement est significativement supérieur à zéro ?

On teste :

$$
H_0 : \mu = 0
$$

Contre :

$$
H_A : \mu > 0
$$

Dans cette version simple, on utilise une statistique de test de type t :

$$
t = \frac{\bar{x} - 0}{s / \sqrt{n}}
$$

Où :

- $\bar{x}$ = moyenne observée du backtest ;
- $s$ = écart-type des rendements ;
- $n$ = nombre d'observations ;
- $s / \sqrt{n}$ = erreur standard de la moyenne.

L'intuition :

> Plus la moyenne observée est grande par rapport au bruit, plus elle devient difficile à expliquer par la chance seule.


In [ ]:

try:
    from scipy import stats

    standard_error = sample_std / np.sqrt(n_days)
    t_stat = sample_mean / standard_error

    # Test unilatéral à droite : P(T >= t_stat)
    p_value = 1 - stats.t.cdf(t_stat, df=n_days - 1)

    alpha = 0.05
    decision = "Rejeter H0 : résultat statistiquement significatif" if p_value < alpha else "Ne pas rejeter H0 : preuve insuffisante"

    test_result = pd.DataFrame({
        "Élément": [
            "Moyenne observée",
            "Erreur standard",
            "t-statistique",
            "p-value",
            "Seuil alpha",
            "Décision"
        ],
        "Valeur": [
            sample_mean,
            standard_error,
            t_stat,
            p_value,
            alpha,
            decision
        ]
    })

    test_result

except ImportError:
    print("scipy n'est pas disponible dans cet environnement.")



## 5. Lecture trader de la p-value

La p-value répond à cette question :

> Si la stratégie n'avait aucun edge réel, quelle serait la probabilité d'observer un résultat au moins aussi bon que celui du backtest ?

Attention :

La p-value ne dit pas directement :

> “Il y a 95 % de chances que ma stratégie marche.”

Elle dit plutôt :

> “Sous l'hypothèse que ma stratégie ne marche pas, ce résultat serait rare ou non.”

C'est une nuance capitale.

Un résultat peut être :

- **profitable mais non significatif** : peut-être de la chance ;
- **profitable et significatif** : plus difficile à expliquer par la chance ;
- **non profitable** : pas besoin de rêver, le test ne sauvera pas la stratégie.



## 6. Estimation ponctuelle : combien la stratégie semble rapporter ?

Après le test, on peut regarder la taille de l'effet.

L'estimation ponctuelle la plus simple est :

$$
\hat{\mu} = \bar{x}
$$

En trading, on annualise souvent :

$$
\hat{\mu}_{annuel} = \bar{x} \times 252
$$

Mais cette valeur seule est dangereuse.

Dire :

> “Ma stratégie rapporte 10 % par an”

est souvent trop affirmatif.

Une formulation plus honnête serait :

> “Sur cet échantillon, la stratégie a montré 10 % annualisé, mais l'incertitude autour de cette estimation est importante.”


In [ ]:

point_estimate_daily = sample_mean
point_estimate_annual = sample_mean * 252

pd.DataFrame({
    "Type d'estimation": [
        "Estimation ponctuelle quotidienne",
        "Estimation ponctuelle annualisée"
    ],
    "Valeur": [
        point_estimate_daily,
        point_estimate_annual
    ]
})



## 7. Estimation par intervalle : quelle fourchette réaliste ?

L'intervalle de confiance ajoute une dimension essentielle : l'incertitude.

Un intervalle de confiance à 95 % pour la moyenne quotidienne peut s'écrire :

$$
\bar{x} \pm t_{critique} \times \frac{s}{\sqrt{n}}
$$

Puis on peut annualiser les bornes.

Interprétation pratique :

> Au lieu de penser en chiffre magique, le trader pense en plage réaliste.

Exemple :

> “Mon estimation centrale est de +8 % annualisé, mais une fourchette plausible est entre -4 % et +20 %.”

Dans ce cas, même avec une estimation centrale positive, le trader doit rester prudent.


In [ ]:

try:
    from scipy import stats

    confidence_level = 0.95
    alpha_ci = 1 - confidence_level

    t_critical = stats.t.ppf(1 - alpha_ci / 2, df=n_days - 1)
    margin_error_daily = t_critical * standard_error

    ci_low_daily = sample_mean - margin_error_daily
    ci_high_daily = sample_mean + margin_error_daily

    ci_low_annual = ci_low_daily * 252
    ci_high_annual = ci_high_daily * 252

    ci_result = pd.DataFrame({
        "Élément": [
            "Estimation annualisée centrale",
            "Borne basse annualisée",
            "Borne haute annualisée",
            "Niveau de confiance"
        ],
        "Valeur": [
            point_estimate_annual,
            ci_low_annual,
            ci_high_annual,
            confidence_level
        ]
    })

    ci_result

except ImportError:
    print("scipy n'est pas disponible dans cet environnement.")


In [ ]:

plt.figure(figsize=(9, 4))
plt.errorbar(
    x=[0],
    y=[point_estimate_annual],
    yerr=[[point_estimate_annual - ci_low_annual], [ci_high_annual - point_estimate_annual]],
    fmt='o',
    capsize=8
)
plt.axhline(0, linestyle='--')
plt.xticks([0], ["Stratégie"])
plt.title("Estimation annualisée avec intervalle de confiance à 95 %")
plt.ylabel("Rendement annualisé estimé")
plt.grid(True)
plt.show()



## 8. Cas pédagogique : profitable ne veut pas toujours dire exploitable

Comparons maintenant deux stratégies :

- Stratégie A : rendement moyen élevé mais peu de données ;
- Stratégie B : rendement moyen plus modeste mais beaucoup plus de données.

Le piège classique :

> Le trader débutant regarde seulement la performance moyenne.

Le trader EBTA regarde aussi :

- la taille de l'échantillon ;
- la variabilité ;
- la significativité ;
- l'intervalle de confiance.


In [ ]:

np.random.seed(7)

strategies = {
    "A_peu_de_donnees": np.random.normal(loc=0.0010, scale=0.015, size=40),
    "B_plus_de_donnees": np.random.normal(loc=0.00035, scale=0.008, size=500)
}

rows = []

for name, r in strategies.items():
    n = len(r)
    mean = np.mean(r)
    std = np.std(r, ddof=1)
    se = std / np.sqrt(n)

    t_stat = mean / se
    p_value = 1 - stats.t.cdf(t_stat, df=n - 1)

    tcrit = stats.t.ppf(0.975, df=n - 1)
    ci_low = (mean - tcrit * se) * 252
    ci_high = (mean + tcrit * se) * 252

    rows.append({
        "Stratégie": name,
        "Nombre de jours": n,
        "Rendement annualisé observé": mean * 252,
        "Volatilité annualisée": std * np.sqrt(252),
        "p-value unilatérale": p_value,
        "Borne basse IC 95 %": ci_low,
        "Borne haute IC 95 %": ci_high,
        "Décision à 5 %": "Rejeter H0" if p_value < 0.05 else "Ne pas rejeter H0"
    })

comparison = pd.DataFrame(rows)
comparison


In [ ]:

plt.figure(figsize=(9, 5))

x = np.arange(len(comparison))
y = comparison["Rendement annualisé observé"].values
lower = y - comparison["Borne basse IC 95 %"].values
upper = comparison["Borne haute IC 95 %"].values - y

plt.errorbar(x, y, yerr=[lower, upper], fmt='o', capsize=8)
plt.axhline(0, linestyle='--')
plt.xticks(x, comparison["Stratégie"], rotation=15)
plt.title("Deux stratégies : performance observée vs incertitude")
plt.ylabel("Rendement annualisé")
plt.grid(True)
plt.show()



## 9. Lecture EBTA de la comparaison

Une stratégie peut afficher une belle moyenne, mais si :

- l'échantillon est trop petit ;
- la volatilité est trop forte ;
- l'intervalle de confiance est énorme ;
- la p-value n'est pas convaincante ;

alors le trader n'a pas encore une preuve solide.

La bonne séquence de pensée est :

1. **Test d'hypothèse** : ai-je des raisons sérieuses de rejeter l'idée que mon edge est nul ?
2. **Estimation** : si oui, quelle taille d'edge semble réaliste ?
3. **Intervalle** : quelle marge d'erreur dois-je accepter ?
4. **Décision business** : est-ce assez robuste pour engager du capital ?



## 10. Le piège : “ne pas rejeter H0” ne veut pas dire “la stratégie est nulle”

C'est un point très important.

Quand le test dit :

> “Ne pas rejeter $H_0$”

cela ne veut pas dire :

> “La stratégie ne marche pas.”

Cela veut dire :

> “Avec les données actuelles, je n'ai pas assez de preuve pour affirmer qu'elle marche.”

La différence est énorme.

Analogie trading :

- Tu vois un setup presque parfait.
- Mais il manque une confirmation importante.
- Tu ne dis pas forcément “ce setup est mauvais”.
- Tu dis : “je n'ai pas assez de confirmation pour le prendre.”

C'est la même logique.



## 11. Résumé ultra-simple

| Outil | Question | Réponse obtenue | Utilité trading |
|---|---|---|---|
| Test d'hypothèse | Est-ce que l'edge existe ? | Décision : rejeter ou non $H_0$ | Filtrer la chance |
| Estimation ponctuelle | Quelle est la performance estimée ? | Une valeur unique | Mesurer l'effet |
| Intervalle de confiance | Quelle est la plage plausible ? | Une fourchette | Quantifier l'incertitude |

La phrase à graver :

> Le test d'hypothèse te protège contre l'illusion.  
> L'estimation t'aide à planifier la réalité.



## 12. Checklist concrète pour un trader

Avant de faire confiance à une stratégie :

1. Vérifier que le backtest est propre : pas de look-ahead bias, pas de survivorship bias, frais inclus.
2. Formuler l'hypothèse nulle : “cette stratégie n'a pas de rendement positif réel”.
3. Calculer une statistique de test pertinente.
4. Obtenir une p-value ou une distribution empirique via bootstrap / permutation.
5. Rejeter $H_0$ seulement si le résultat est suffisamment rare sous le hasard.
6. Ensuite seulement, estimer la taille de l'edge.
7. Préférer un intervalle de confiance à une simple moyenne.
8. Décider si l'edge estimé est suffisant après frais, slippage et drawdowns.



## 13. Conclusion

Les deux types d'inférence statistique ne répondent pas à la même question.

Le **test d'hypothèse** est un garde-fou :

> “Est-ce que je suis probablement face à autre chose que de la chance ?”

L'**estimation** est un outil de dimensionnement :

> “Si l'avantage existe, combien vaut-il probablement ?”

En trading, mélanger ces deux questions crée beaucoup d'illusions.

Un backtest peut être impressionnant mais non significatif.  
Un edge peut exister mais être trop petit pour survivre aux frais.  
Une moyenne peut être positive mais avoir un intervalle de confiance trop large.

La maturité statistique consiste à dire :

> “Je ne veux pas seulement gagner dans le passé.  
> Je veux savoir si mon résultat est crédible, mesurable et exploitable.”


## Annexe — Fiche de Lecture & Source Originale

> [!NOTE]
> **Source :** David Aronson, *Evidence-Based Technical Analysis (EBTA)*  
> **Référence :** TWO TYPES OF STATISTICAL INFERENCE (Pages 217–218 ; Audiobook Transcriptions 135-136, 168)

---

### 📌 Idées Clés
* **Division de l'Inférence Statistique** : Elle se subdivise en deux procédures distinctes :
  1. **Le test d'hypothèse** (procédure de décision binaire) : il détermine si un effet est présent ou non (filtrer la chance).
  2. **L'estimation de paramètre** (procédure quantitative) : elle cherche à déterminer la taille ou l'ampleur de cet effet.
* **Le point commun** : Ces deux méthodes s'intéressent à la valeur inconnue d'un paramètre de population (la performance future réelle de la stratégie) à partir d'un échantillon limité (le backtest).

---

### 💬 Citation Directe
> « *Statistical inference encompasses two procedures: hypothesis testing and parameter estimation. Both are concerned with the unknown value of a population parameter. [...] A hypothesis test tells us if an effect is present or not, whereas an estimate tells us about the size of an effect.* » (Page 217)

---

### 🌐 Vision Macro
L'enjeu pour le trader est de naviguer dans l'incertitude sans se fier à des "comptes de fées". Comme on ne peut pas observer l'intégralité du futur, on doit faire un **"saut inductif"**.
* **Le test d'hypothèse** agit comme un garde-fou contre l'illusion (la chance).
* **L'estimation** permet de planifier rationnellement l'avenir financier en quantifiant ce que la stratégie peut réellement rapporter.

---

### 🔍 Vision Micro

#### 1. Le Test d'Hypothèse (*Hypothesis Testing*)
* **Mécanisme** : On commence par une conjecture (souvent l'Hypothèse Nulle $H_0$ : "*la règle n'a aucun talent*").
* **Logique** : On utilise la falsification. Si le résultat observé est trop improbable sous l'hypothèse $H_0$, on rejette cette dernière au profit de l'hypothèse alternative $H_A$.
* **Résultat** : Une conclusion de type binaire : "*Rejeter*" ou "*Ne pas rejeter*" $H_0$.

#### 2. L'Estimation de Paramètre (*Parameter Estimation*)
* **Point Estimate (Estimation ponctuelle)** : Une valeur unique (comme la moyenne des rendements du backtest) utilisée pour deviner le rendement futur.
* **Interval Estimate (Estimation par intervalle)** : Une plage de valeurs (ex: entre 5% et 15%) assortie d'un niveau de probabilité (ex: 95%).
* **Avantage** : L'intervalle de confiance est beaucoup plus riche et informatif car il combine l'estimation du profit avec la mesure de l'incertitude (variabilité).

---

### 💡 Résumé Simplifié (Analogie)
> Imaginez que vous testez un nouveau moteur :
> * **Le test d'hypothèse** répond à la question : *« Est-ce que le moteur démarre vraiment grâce à sa technologie ou est-ce un hasard ? »* (**Le filtre de sécurité**).
> * **L'estimation** répond à : *« À quelle vitesse va-t-il me propulser et avec quelle marge d'erreur ? »* (**L'outil de mesure de performance**).

---

### 🛠️ Actions Concrètes pour le Trader
1. **Ne jamais estimer avant de tester** : Ne perdez pas de temps à calculer vos gains futurs potentiels si votre stratégie n'a pas d'abord prouvé sa signification statistique ($p\text{-value} < 0.05$).
2. **Utiliser l'estimation comme appoint** : Dans les études EBTA, l'estimation est utilisée comme un complément aux tests d'hypothèse pour affiner la compréhension de la règle.
3. **Privilégier les intervalles** : Ne dites jamais *« ma stratégie rapporte 10 % »*, dites *« il y a 95 % de chances que ma stratégie rapporte entre 5 % et 15 % »* pour rester ancré dans la réalité statistique.

---

### ⚠️ À Retenir Absolument
* **Test d'hypothèse** = Existe-t-il un avantage réel ?
* **Estimation** = Quelle est l'importance de cet avantage ?
* Toute inférence est un **saut inductif** et comporte donc un risque d'erreur.
* **L'intervalle de confiance** est la forme la plus riche d'estimation car il quantifie le doute.
* Sans ces outils, le trader est victime de ses propres **biais cognitifs** (ex: biais de confirmation).
